# **Comparative Study of TF-IDF and Word2Vec for Text Similarity and Content Recommendation**

**Importing Required Libraries**

In [ ]:
import pandas as pd
import numpy as np
import re
import nltk
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from gensim.models import Word2Vec

# Download NLTK resources for text processing
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

import warnings
warnings.filterwarnings("ignore")


In [ ]:
df = pd.read_csv('bbc-text.csv')
print("Dataset Loaded Successfully!")
display(df.head())

 **1. Data Preprocessing & Exploratory Data Analysis**

In [ ]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    # 1. Remove URLs, special characters, and numbers
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'[^a-zA-Z\s]', '', text)

    # 2. Convert to lowercase
    text = text.lower()

    # 3. Tokenization
    tokens = word_tokenize(text)

    # 4. Stopword removal and Lemmatization
    # We also filter out very short words (length < 3)
    tokens = [lemmatizer.lemmatize(word) for word in tokens if word not in stop_words and len(word) > 2]

    return " ".join(tokens)

# Apply preprocessing to the entire dataset
df['clean_text'] = df['text'].apply(preprocess_text)

# Function to visualize Top N-grams
from sklearn.feature_extraction.text import CountVectorizer

def plot_ngrams(corpus, n_range, title, color):
    vec = CountVectorizer(ngram_range=n_range, stop_words='english').fit(corpus)
    bag_of_words = vec.transform(corpus)
    sum_words = bag_of_words.sum(axis=0)
    words_freq = sorted([(word, sum_words[0, idx]) for word, idx in vec.vocabulary_.items()], key=lambda x: x[1], reverse=True)[:15]
    ngram_df = pd.DataFrame(words_freq, columns=['Text', 'Count'])
    sns.barplot(x='Count', y='Text', data=ngram_df, palette=color)
    plt.title(title)

# Plotting Unigrams, Bigrams, and Trigrams
plt.figure(figsize=(10, 15))
plt.subplot(3, 1, 1); plot_ngrams(df['clean_text'], (1,1), "Top 15 Unigrams", "Blues_d")
plt.subplot(3, 1, 2); plot_ngrams(df['clean_text'], (2,2), "Top 15 Bigrams", "Reds_d")
plt.subplot(3, 1, 3); plot_ngrams(df['clean_text'], (3,3), "Top 15 Trigrams", "Greens_d")
plt.tight_layout()
plt.show()

# Generate Word Cloud for visual word distribution
wordcloud = WordCloud(width=800, height=400, background_color='white').generate(" ".join(df['clean_text']))
plt.figure(figsize=(10, 5))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.title('Word Cloud of BBC Articles')
plt.show()

**2. TF-IDF Based Article Recommendation**

In [ ]:
from IPython.display import display

# Generate TF-IDF Matrix
tfidf_vectorizer = TfidfVectorizer()
tfidf_matrix = tfidf_vectorizer.fit_transform(df['clean_text'])

def get_recommendations_tfidf(query, vectorizer, matrix, original_df, top_n=5):
    # Preprocess user input
    query_clean = preprocess_text(query)
    # Vectorize query
    query_vec = vectorizer.transform([query_clean])

    # Calculate Similarity
    scores = cosine_similarity(query_vec, matrix).flatten()

    # Sort and get top indices
    top_indices = scores.argsort()[-top_n:][::-1]

    # Format results
    results = original_df.iloc[top_indices].copy()
    results['similarity_score'] = scores[top_indices]

    # Returning only first 200 characters of text for cleaner display
    results['text_snippet'] = results['text'].apply(lambda x: x[:200] + "...")
    return results[['category', 'text_snippet', 'similarity_score']]

# Test execution
user_query = "The future of mobile technology and digital gadgets in consumer electronics."
recommendations = get_recommendations_tfidf(user_query, tfidf_vectorizer, tfidf_matrix, df)

print(f"User Query: {user_query}")
print("\n--- TF-IDF Top 5 Recommendations ---")
display(recommendations)

**3. Word2Vec Based Article Recommendation (CBOW & Skip-gram)**

In [ ]:
from IPython.display import display, HTML

# 1. Prepare tokenized sentences
tokenized_docs = [text.split() for text in df['clean_text']]

# 2. Train Word2Vec models
# CBOW (Continuous Bag of Words)
model_cbow = Word2Vec(sentences=tokenized_docs, vector_size=100, window=5, min_count=1, sg=0)
# Skip-gram
model_sg = Word2Vec(sentences=tokenized_docs, vector_size=100, window=5, min_count=1, sg=1)

# 3. Helper function for Average Word Embeddings
def get_avg_embedding(tokens, model, size=100):
    vecs = [model.wv[word] for word in tokens if word in model.wv]
    return np.mean(vecs, axis=0) if vecs else np.zeros(size)

# 4. Generate document vectors
df_vecs_cbow = np.array([get_avg_embedding(t, model_cbow) for t in tokenized_docs])
df_vecs_sg = np.array([get_avg_embedding(t, model_sg) for t in tokenized_docs])

# 5. Recommendation function with clean display
def recommend_w2v_display(query, model, doc_vectors, original_df, model_name, top_n=5):
    query_tokens = preprocess_text(query).split()
    query_vec = get_avg_embedding(query_tokens, model).reshape(1, -1)

    # Cosine Similarity calculation
    scores = cosine_similarity(query_vec, doc_vectors).flatten()
    top_indices = scores.argsort()[-top_n:][::-1]

    results = original_df.iloc[top_indices].copy()
    results['similarity_score'] = scores[top_indices]

    # Snippet for cleaner view
    results['text_snippet'] = results['text'].apply(lambda x: x[:150] + "...")

    print(f"\n--- {model_name} Top 5 Recommendations ---")
    display(results[['category', 'text_snippet', 'similarity_score']])

# --- EXECUTION ---
user_query = "The future of mobile technology and digital gadgets in consumer electronics."
print(f"User Query: {user_query}")

# Show CBOW results
recommend_w2v_display(user_query, model_cbow, df_vecs_cbow, df, "Word2Vec CBOW")

# Show Skip-gram results
recommend_w2v_display(user_query, model_sg, df_vecs_sg, df, "Word2Vec Skip-gram")

**4. Comparative Analysis: TF-IDF vs. Word2Vec**

**1.TF-IDF vs. Word2Vec Results:**

TF-IDF is best for exact keyword matching. It successfully found articles containing specific words like "mobile" or "gadgets."

Word2Vec captures context. Even if an article used the word "device" instead of "gadget," Word2Vec could recommend it because it understands they are semantically related.

Winner: For this specific dataset, TF-IDF provides very clear, relevant results. However, Word2Vec is "smarter" at finding related topics that don't share exact words.

**2. Limitations:**

**TF-IDF:** It ignores word order and cannot understand synonyms (e.g., it treats "phone" and "mobile" as completely different things).

**Word2Vec:** It requires a lot of data to train well. Averaging word vectors to represent a whole document can sometimes "wash out" the importance of specific unique keywords.

**CBOW vs Skip-gram:** CBOW is faster and works well with frequent words, while Skip-gram is better at representing rare words and capturing deeper context.
